In [ ]:
"""
Tech Challenge - Fase 1 - Challenge B

Script unico para executar o pipeline completo pedido no PDF:
- carregamento e discussao do dataset;
- exploracao, estatisticas descritivas e visualizacoes;
- limpeza e pre-processamento;
- analise de correlacao;
- treino, validacao e teste com multiplos modelos de classificacao;
- avaliacao com accuracy, recall e F1-score;
- interpretacao com feature importance e SHAP quando disponivel;
- geracao de artefatos, relatorio, README e Dockerfile.

Uso:
    python techchallengeB.py
    python techchallengeB.py --data "C:/caminho/para/data.csv" --output outputs_techchallenge_b
"""

from __future__ import annotations

import argparse
import json
import math
import pickle
import textwrap
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.base import BaseEstimator
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier


warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# Configuracoes centrais do problema: coluna alvo original, alvo binario
# criado pelo script e codigos usados para as classes.
TARGET_COLUMN = "Dangerous"
TARGET_NAME = "dangerous_binary"
POSITIVE_LABEL = 1
NEGATIVE_LABEL = 0

# Caminhos testados automaticamente quando o usuario nao informa --data.
DEFAULT_DATASET_CANDIDATES = [
    Path("data.csv"),
    Path(r"C:\Users\Desktop\Downloads\animal_condition_dataset\data.csv"),
]


@dataclass
class SplitData:
    x_train: pd.DataFrame
    x_val: pd.DataFrame
    x_test: pd.DataFrame
    y_train: pd.Series
    y_val: pd.Series
    y_test: pd.Series


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Pipeline completo de Machine Learning para o Tech Challenge B."
    )
    parser.add_argument("--data", type=str, default=None)
    parser.add_argument("--output", type=str, default="outputs_techchallenge_b")
    parser.add_argument("--random-state", type=int, default=42)
    parser.add_argument("--validation-size", type=float, default=0.20)
    parser.add_argument("--test-size", type=float, default=0.20)
    parser.add_argument("--top-n-features", type=int, default=25)

    args, _ = parser.parse_known_args()
    return args


def resolve_dataset_path(path_arg: Optional[str]) -> Path:
    # Decide qual CSV sera usado: primeiro respeita --data; se nao vier,
    # tenta os caminhos padrao definidos acima.
    if path_arg:
        path = Path(path_arg).expanduser()
        if path.exists():
            return path
        raise FileNotFoundError(f"Dataset nao encontrado: {path}")

    for candidate in DEFAULT_DATASET_CANDIDATES:
        if candidate.exists():
            return candidate

    searched = ", ".join(str(p) for p in DEFAULT_DATASET_CANDIDATES)
    raise FileNotFoundError(
        "Dataset nao encontrado. Informe --data. Caminhos testados: " + searched
    )


def create_output_dirs(base: Path) -> Dict[str, Path]:
    # Cria a estrutura onde graficos, tabelas, modelos, relatorios e docs serao salvos.
    dirs = {
        "base": base,
        "figures": base / "figures",
        "tables": base / "tables",
        "models": base / "models",
        "reports": base / "reports",
        "docs": base / "docs",
    }
    for directory in dirs.values():
        directory.mkdir(parents=True, exist_ok=True)
    return dirs


def save_json(data: dict, path: Path) -> None:
    # Salva dicionarios em JSON legivel para facilitar auditoria dos resultados.
    with path.open("w", encoding="utf-8") as file:
        json.dump(data, file, indent=2, ensure_ascii=False)


def normalize_text(value: object) -> object:
    # Padroniza textos categoricos para reduzir duplicidade causada por caixa,
    # espacos extras e pequenos erros de digitacao.
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower()
    text = " ".join(text.split())
    replacements = {
        "seizuers": "seizures",
        "anorexia": "loss of appetite",
        "poor appetite": "loss of appetite",
        "tiredness": "fatigue",
    }
    return replacements.get(text, text)


def load_dataset(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)

    # normaliza nomes das colunas
    df.columns = [str(col).strip() for col in df.columns]

    rename_map = {
        "Animal": "AnimalName",
        "Symptom 1": "symptoms1",
        "Symptom 2": "symptoms2",
        "Symptom 3": "symptoms3",
        "Symptom 4": "symptoms4",
        "Symptom 5": "symptoms5",
    }

    df = df.rename(columns=rename_map)

    required = {
        "AnimalName",
        "symptoms1",
        "symptoms2",
        "symptoms3",
        "symptoms4",
        "symptoms5",
        TARGET_COLUMN
    }

    missing = sorted(required.difference(df.columns))

    if missing:
        raise ValueError(f"Colunas obrigatorias ausentes no CSV: {missing}")

    return df


def clean_dataset(df_raw: pd.DataFrame) -> pd.DataFrame:
    # Executa a limpeza principal: remove duplicatas, normaliza textos,
    # converte o alvo Yes/No para 1/0 e cria features auxiliares.
    df = df_raw.copy()
    df = df.drop_duplicates().reset_index(drop=True)

    text_columns = ["AnimalName", "symptoms1", "symptoms2", "symptoms3", "symptoms4", "symptoms5"]
    for col in text_columns:
        df[col] = df[col].map(normalize_text)
        df[col] = df[col].fillna("unknown")

    target_map = {
        "yes": POSITIVE_LABEL,
        "y": POSITIVE_LABEL,
        "true": POSITIVE_LABEL,
        "1": POSITIVE_LABEL,
        "no": NEGATIVE_LABEL,
        "n": NEGATIVE_LABEL,
        "false": NEGATIVE_LABEL,
        "0": NEGATIVE_LABEL,
    }
    df[TARGET_COLUMN] = df[TARGET_COLUMN].map(normalize_text)
    df[TARGET_NAME] = df[TARGET_COLUMN].map(target_map)
    df = df.dropna(subset=[TARGET_NAME]).copy()
    df[TARGET_NAME] = df[TARGET_NAME].astype(int)

    symptom_cols = [f"symptoms{i}" for i in range(1, 6)]
    df["symptoms_text"] = df[symptom_cols].agg(" ".join, axis=1)
    df["unique_symptom_count"] = df[symptom_cols].nunique(axis=1)
    df["unknown_symptom_count"] = (df[symptom_cols] == "unknown").sum(axis=1)
    df["symptom_text_length"] = df["symptoms_text"].str.len()
    return df


def summarize_dataset(df_raw: pd.DataFrame, df: pd.DataFrame, dirs: Dict[str, Path]) -> dict:
    # Gera tabelas resumidas da base para apoiar a exploracao e alimentar o relatorio.
    symptom_cols = [f"symptoms{i}" for i in range(1, 6)]
    all_symptoms = pd.concat([df[col] for col in symptom_cols], ignore_index=True)

    summary = {
        "raw_rows": int(len(df_raw)),
        "clean_rows": int(len(df)),
        "raw_columns": list(df_raw.columns),
        "duplicates_removed": int(len(df_raw) - len(df_raw.drop_duplicates())),
        "target_distribution": df[TARGET_COLUMN].value_counts().to_dict(),
        "animal_distribution": df["AnimalName"].value_counts().to_dict(),
        "top_symptoms": all_symptoms.value_counts().head(30).to_dict(),
        "missing_values_raw": df_raw.isna().sum().to_dict(),
        "numeric_describe": df[
            ["unique_symptom_count", "unknown_symptom_count", "symptom_text_length"]
        ].describe().round(3).to_dict(),
    }
    save_json(summary, dirs["tables"] / "dataset_summary.json")

    pd.DataFrame({"missing_values": df_raw.isna().sum()}).to_csv(
        dirs["tables"] / "missing_values.csv", encoding="utf-8"
    )
    df[TARGET_COLUMN].value_counts().rename_axis("target").reset_index(name="count").to_csv(
        dirs["tables"] / "target_distribution.csv", index=False, encoding="utf-8"
    )
    all_symptoms.value_counts().rename_axis("symptom").reset_index(name="count").to_csv(
        dirs["tables"] / "symptom_frequency.csv", index=False, encoding="utf-8"
    )
    return summary


def bar_plot(series: pd.Series, title: str, xlabel: str, ylabel: str, path: Path, top_n: int = 20) -> None:
    # Funcao utilitaria para criar graficos de barras horizontais reutilizados na EDA.
    data = series.head(top_n).sort_values(ascending=True)
    height = max(4.0, min(12.0, 0.35 * len(data) + 1.5))
    plt.figure(figsize=(10, height))
    plt.barh(data.index.astype(str), data.values, color="#2F6F73")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()


def create_eda_plots(df_raw: pd.DataFrame, df: pd.DataFrame, dirs: Dict[str, Path]) -> List[Path]:
    # Cria graficos de EDA: alvo, animais, sintomas, features numericas e ausentes.
    figures: List[Path] = []
    symptom_cols = [f"symptoms{i}" for i in range(1, 6)]
    all_symptoms = pd.concat([df[col] for col in symptom_cols], ignore_index=True)

    path = dirs["figures"] / "01_target_distribution.png"
    bar_plot(df[TARGET_COLUMN].value_counts(), "Distribuicao do alvo", "Quantidade", "Classe", path)
    figures.append(path)

    path = dirs["figures"] / "02_top_animals.png"
    bar_plot(df["AnimalName"].value_counts(), "Animais mais frequentes", "Quantidade", "Animal", path)
    figures.append(path)

    path = dirs["figures"] / "03_top_symptoms.png"
    bar_plot(all_symptoms.value_counts(), "Sintomas mais frequentes", "Quantidade", "Sintoma", path, top_n=25)
    figures.append(path)

    path = dirs["figures"] / "04_numeric_features_by_target.png"
    numeric_cols = ["unique_symptom_count", "unknown_symptom_count", "symptom_text_length"]
    fig, axes = plt.subplots(1, len(numeric_cols), figsize=(14, 4))
    for ax, col in zip(axes, numeric_cols):
        df.boxplot(column=col, by=TARGET_COLUMN, ax=ax, grid=False)
        ax.set_title(col)
        ax.set_xlabel("Dangerous")
        ax.set_ylabel("Valor")
    fig.suptitle("Features numericas por classe")
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()
    figures.append(path)

    path = dirs["figures"] / "05_missing_values.png"
    missing = df_raw.isna().sum().sort_values(ascending=True)
    plt.figure(figsize=(9, 4))
    plt.barh(missing.index.astype(str), missing.values, color="#7A4E8A")
    plt.title("Valores ausentes antes da limpeza")
    plt.xlabel("Quantidade")
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()
    figures.append(path)
    return figures


def cramers_v(x: pd.Series, y: pd.Series) -> float:
    # Calcula Cramer's V, medida de associacao entre variaveis categoricas.
    confusion = pd.crosstab(x, y)
    if confusion.empty:
        return 0.0
    observed = confusion.to_numpy(dtype=float)
    total = observed.sum()
    if total == 0:
        return 0.0
    row_sums = observed.sum(axis=1, keepdims=True)
    col_sums = observed.sum(axis=0, keepdims=True)
    expected = row_sums @ col_sums / total
    with np.errstate(divide="ignore", invalid="ignore"):
        chi2 = np.nansum((observed - expected) ** 2 / expected)
    n = total
    r, k = observed.shape
    denominator = min(k - 1, r - 1)
    if denominator <= 0:
        return 0.0
    return float(math.sqrt((chi2 / n) / denominator))


def correlation_analysis(df: pd.DataFrame, dirs: Dict[str, Path], top_n: int) -> Tuple[pd.DataFrame, List[Path]]:
    # Mede relacoes entre features e alvo com Pearson nas dummies e Cramer's V nas categoricas.
    figures: List[Path] = []
    feature_cols = ["AnimalName", "symptoms1", "symptoms2", "symptoms3", "symptoms4", "symptoms5"]
    encoded = pd.get_dummies(df[feature_cols], prefix=feature_cols)
    encoded[TARGET_NAME] = df[TARGET_NAME].values
    corr = encoded.corr(numeric_only=True)[TARGET_NAME].drop(TARGET_NAME)
    corr_table = corr.sort_values(key=lambda s: s.abs(), ascending=False).reset_index()
    corr_table.columns = ["encoded_feature", "pearson_corr_with_target"]
    corr_table.to_csv(dirs["tables"] / "encoded_feature_correlations.csv", index=False, encoding="utf-8")

    categorical_association = pd.DataFrame(
        {
            "feature": feature_cols,
            "cramers_v_with_target": [cramers_v(df[col], df[TARGET_NAME]) for col in feature_cols],
        }
    ).sort_values("cramers_v_with_target", ascending=False)
    categorical_association.to_csv(
        dirs["tables"] / "categorical_association_cramers_v.csv", index=False, encoding="utf-8"
    )

    path = dirs["figures"] / "06_top_correlations.png"
    top = corr_table.head(top_n).iloc[::-1]
    colors = np.where(top["pearson_corr_with_target"] >= 0, "#2F6F73", "#B85C5C")
    plt.figure(figsize=(11, max(5, 0.35 * len(top))))
    plt.barh(top["encoded_feature"], top["pearson_corr_with_target"], color=colors)
    plt.axvline(0, color="black", linewidth=0.8)
    plt.title("Features codificadas mais correlacionadas com Dangerous")
    plt.xlabel("Correlacao de Pearson")
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()
    figures.append(path)

    path = dirs["figures"] / "07_cramers_v.png"
    top_assoc = categorical_association.iloc[::-1]
    plt.figure(figsize=(9, 4))
    plt.barh(top_assoc["feature"], top_assoc["cramers_v_with_target"], color="#4C6B9A")
    plt.title("Associacao categorica com o alvo - Cramer's V")
    plt.xlabel("Cramer's V")
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()
    figures.append(path)
    return corr_table, figures


def make_one_hot_encoder() -> OneHotEncoder:
    # Mantem compatibilidade com versoes novas e antigas do scikit-learn.
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocessor(categorical_cols: List[str], numeric_cols: List[str]) -> ColumnTransformer:
    # Monta o pre-processamento: one-hot para categoricas e escala para numericas.
    return ColumnTransformer(
        transformers=[
            ("categorical", make_one_hot_encoder(), categorical_cols),
            ("numeric", StandardScaler(), numeric_cols),
        ],
        remainder="drop",
        sparse_threshold=0.0,
    )


def split_data(
    df: pd.DataFrame,
    feature_cols: List[str],
    target_col: str,
    validation_size: float,
    test_size: float,
    random_state: int,
) -> SplitData:
    # Separa a base em treino, validacao e teste preservando a proporcao das classes.
    if validation_size <= 0 or test_size <= 0 or validation_size + test_size >= 0.8:
        raise ValueError("Use validation_size e test_size positivos, com soma menor que 0.8.")

    x = df[feature_cols]
    y = df[target_col]
    temp_size = validation_size + test_size
    x_train, x_temp, y_train, y_temp = train_test_split(
        x, y, test_size=temp_size, stratify=y, random_state=random_state
    )
    val_fraction_inside_temp = validation_size / temp_size
    x_val, x_test, y_val, y_test = train_test_split(
        x_temp,
        y_temp,
        test_size=1 - val_fraction_inside_temp,
        stratify=y_temp,
        random_state=random_state,
    )
    return SplitData(x_train, x_val, x_test, y_train, y_val, y_test)


def build_models(
    categorical_cols: List[str], numeric_cols: List[str], random_state: int
) -> Dict[str, Pipeline]:
    # Define os modelos comparados. Cada um recebe seu proprio Pipeline.
    models: Dict[str, BaseEstimator] = {
        "logistic_regression": LogisticRegression(
            max_iter=2000, class_weight="balanced", solver="liblinear", random_state=random_state
        ),
        "decision_tree": DecisionTreeClassifier(
            max_depth=10, min_samples_leaf=3, class_weight="balanced", random_state=random_state
        ),
        "random_forest": RandomForestClassifier(
            n_estimators=350,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=random_state,
            n_jobs=-1,
        ),
        "knn": KNeighborsClassifier(n_neighbors=7, weights="distance"),
    }
    return {
        name: Pipeline(
            [
                ("preprocess", build_preprocessor(categorical_cols, numeric_cols)),
                ("model", model),
            ]
        )
        for name, model in models.items()
    }


def prediction_scores(model: Pipeline, x: pd.DataFrame) -> Optional[np.ndarray]:
    # Retorna probabilidade/pontuacao da classe positiva quando o modelo permite.
    if hasattr(model, "predict_proba"):
        return model.predict_proba(x)[:, 1]
    if hasattr(model, "decision_function"):
        scores = model.decision_function(x)
        return (scores - scores.min()) / (scores.max() - scores.min() + 1e-12)
    return None


def evaluate_model(model: Pipeline, x: pd.DataFrame, y: pd.Series) -> dict:
    # Calcula as metricas principais, com foco em recall e F1 da classe perigosa.
    pred = model.predict(x)
    scores = prediction_scores(model, x)
    result = {
        "accuracy": accuracy_score(y, pred),
        "precision_yes": precision_score(y, pred, pos_label=POSITIVE_LABEL, zero_division=0),
        "recall_yes": recall_score(y, pred, pos_label=POSITIVE_LABEL, zero_division=0),
        "f1_yes": f1_score(y, pred, pos_label=POSITIVE_LABEL, zero_division=0),
        "f1_weighted": f1_score(y, pred, average="weighted", zero_division=0),
    }
    if scores is not None and len(np.unique(y)) == 2:
        result["roc_auc"] = roc_auc_score(y, scores)
    else:
        result["roc_auc"] = np.nan
    return result


def train_and_select_models(
    split: SplitData,
    categorical_cols: List[str],
    numeric_cols: List[str],
    random_state: int,
    dirs: Dict[str, Path],
) -> Tuple[str, Pipeline, pd.DataFrame]:
    # Treina todos os modelos, avalia na validacao e escolhe o melhor por recall/F1.
    models = build_models(categorical_cols, numeric_cols, random_state)

    records = []
    for name, model in models.items():
        print(f"Treinando modelo: {name}")
        model.fit(split.x_train, split.y_train)
        val_metrics = evaluate_model(model, split.x_val, split.y_val)
        records.append({"model": name, **val_metrics})

    validation_table = pd.DataFrame(records).sort_values(
        ["recall_yes", "f1_yes", "accuracy"], ascending=False
    )
    validation_table.to_csv(dirs["tables"] / "validation_metrics.csv", index=False, encoding="utf-8")
    best_name = str(validation_table.iloc[0]["model"])
    best_model = models[best_name]
    return best_name, best_model, validation_table


def save_confusion_matrix(y_true: pd.Series, y_pred: np.ndarray, title: str, path: Path) -> None:
    # Gera matriz de confusao para visualizar acertos e erros entre No e Yes.
    matrix = confusion_matrix(y_true, y_pred, labels=[NEGATIVE_LABEL, POSITIVE_LABEL])
    display = ConfusionMatrixDisplay(
        confusion_matrix=matrix, display_labels=["No", "Yes"]
    )
    display.plot(cmap="Blues", values_format="d")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()


def evaluate_best_model(
    model_name: str,
    model: Pipeline,
    split: SplitData,
    dirs: Dict[str, Path],
) -> Tuple[pd.DataFrame, dict, List[Path]]:
    # Avalia o modelo escolhido em treino, validacao e teste, salvando metricas e graficos.
    figures: List[Path] = []
    datasets = {
        "train": (split.x_train, split.y_train),
        "validation": (split.x_val, split.y_val),
        "test": (split.x_test, split.y_test),
    }

    records = []
    reports = {}
    for dataset_name, (x_part, y_part) in datasets.items():
        pred = model.predict(x_part)
        metrics = evaluate_model(model, x_part, y_part)
        records.append({"dataset": dataset_name, "model": model_name, **metrics})
        reports[dataset_name] = classification_report(
            y_part,
            pred,
            target_names=["No - nao perigoso", "Yes - perigoso"],
            output_dict=True,
            zero_division=0,
        )
        path = dirs["figures"] / f"08_confusion_matrix_{dataset_name}.png"
        save_confusion_matrix(y_part, pred, f"Matriz de confusao - {dataset_name}", path)
        figures.append(path)

    metrics_table = pd.DataFrame(records)
    metrics_table.to_csv(dirs["tables"] / "best_model_metrics.csv", index=False, encoding="utf-8")
    save_json(reports, dirs["tables"] / "classification_reports.json")
    return metrics_table, reports, figures


def get_feature_names(model: Pipeline, original_cols: List[str]) -> np.ndarray:
    # Recupera os nomes finais das features apos o pre-processamento.
    preprocessor = model.named_steps["preprocess"]
    try:
        return preprocessor.get_feature_names_out(original_cols)
    except Exception:
        names: List[str] = []
        categorical_encoder = preprocessor.named_transformers_.get("categorical")
        if categorical_encoder is not None and hasattr(categorical_encoder, "get_feature_names_out"):
            cat_cols = preprocessor.transformers_[0][2]
            names.extend(categorical_encoder.get_feature_names_out(cat_cols).tolist())
        numeric_cols = preprocessor.transformers_[1][2]
        names.extend([f"numeric__{col}" for col in numeric_cols])
        return np.array(names)


def model_feature_importance(
    model: Pipeline,
    feature_cols: List[str],
    split: SplitData,
    dirs: Dict[str, Path],
    top_n: int,
) -> Tuple[pd.DataFrame, List[Path]]:
    # Interpreta o modelo usando importancia nativa, coeficientes ou permutation importance.
    figures: List[Path] = []
    estimator = model.named_steps["model"]
    feature_names = get_feature_names(model, feature_cols)

    if hasattr(estimator, "feature_importances_"):
        values = estimator.feature_importances_
        method = "model_feature_importances"
    elif hasattr(estimator, "coef_"):
        values = np.ravel(np.abs(estimator.coef_))
        method = "absolute_model_coefficients"
    else:
        print("Modelo sem importance nativa. Calculando permutation importance no teste.")
        permutation = permutation_importance(
            model,
            split.x_test,
            split.y_test,
            n_repeats=12,
            random_state=42,
            scoring="f1",
            n_jobs=-1,
        )
        feature_names = np.array(feature_cols)
        values = permutation.importances_mean
        method = "permutation_importance"

    length = min(len(feature_names), len(values))
    table = pd.DataFrame(
        {
            "feature": feature_names[:length],
            "importance": values[:length],
            "method": method,
        }
    ).sort_values("importance", ascending=False)
    table.to_csv(dirs["tables"] / "feature_importance.csv", index=False, encoding="utf-8")

    path = dirs["figures"] / "09_feature_importance.png"
    top = table.head(top_n).iloc[::-1]
    plt.figure(figsize=(11, max(5, 0.35 * len(top))))
    plt.barh(top["feature"], top["importance"], color="#5A7D3A")
    plt.title(f"Top {len(top)} features - {method}")
    plt.xlabel("Importancia")
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()
    figures.append(path)
    return table, figures


def try_generate_shap(
    model: Pipeline,
    split: SplitData,
    feature_cols: List[str],
    dirs: Dict[str, Path],
    top_n: int,
) -> Tuple[Optional[Path], str]:
    # Tenta gerar explicabilidade com SHAP; se falhar, o pipeline continua normalmente.
    try:
        import shap  # type: ignore
    except Exception:
        return None, "SHAP nao instalado. Foi usado feature importance/permutation importance como fallback."

    try:
        transformed = model.named_steps["preprocess"].transform(split.x_test)
        feature_names = get_feature_names(model, feature_cols)
        if hasattr(transformed, "toarray"):
            transformed = transformed.toarray()
        sample_size = min(200, transformed.shape[0])
        x_sample = transformed[:sample_size]
        estimator = model.named_steps["model"]

        explainer = shap.Explainer(estimator, x_sample, feature_names=feature_names)
        shap_values = explainer(x_sample)

        plt.figure()
        shap.plots.bar(shap_values, max_display=top_n, show=False)
        path = dirs["figures"] / "10_shap_summary_bar.png"
        plt.tight_layout()
        plt.savefig(path, dpi=160, bbox_inches="tight")
        plt.close()
        return path, "SHAP executado com sucesso para uma amostra do conjunto de teste."
    except Exception as exc:
        message = (
            "SHAP estava instalado, mas falhou neste modelo/ambiente. "
            f"Fallback mantido com feature importance. Erro: {exc}"
        )
        return None, message


def write_prediction_example(model: Pipeline, split: SplitData, dirs: Dict[str, Path]) -> dict:
    # Salva um exemplo de predicao para demonstrar o uso do modelo em triagem.
    sample = split.x_test.iloc[[0]].copy()
    pred = int(model.predict(sample)[0])
    scores = prediction_scores(model, sample)
    probability = float(scores[0]) if scores is not None else None
    result = {
        "input": sample.iloc[0].to_dict(),
        "prediction_binary": pred,
        "prediction_label": "Yes - perigoso" if pred == POSITIVE_LABEL else "No - nao perigoso",
        "probability_yes": probability,
        "warning": "A previsao e apenas apoio a triagem. A decisao final deve ser do profissional de saude.",
    }
    save_json(result, dirs["tables"] / "prediction_example.json")
    return result


def write_readme_and_dockerfile(dirs: Dict[str, Path]) -> Tuple[Path, Path, Path]:
    # Gera README, Dockerfile e roteiro sugerido para o video de demonstracao.
    readme = dirs["docs"] / "README.md"
    dockerfile = dirs["docs"] / "Dockerfile"
    video_script = dirs["docs"] / "roteiro_video_demo.md"

    readme.write_text(
        textwrap.dedent(
            """
            # Tech Challenge B - Pipeline de IA

            Este projeto executa um pipeline completo de Machine Learning para classificar
            se um caso deve ser marcado como perigoso a partir de animal e sintomas.

            ## Execucao local

            ```bash
            python techchallengeB.py --data data.csv --output outputs_techchallenge_b
            ```

            ## Principais artefatos gerados

            - `reports/relatorio_tecnico.md`
            - `reports/relatorio_tecnico.pdf`
            - `tables/*.csv` e `tables/*.json`
            - `figures/*.png`
            - `models/best_model.pkl`

            ## Docker

            Copie `techchallengeB.py`, `data.csv` e este Dockerfile para a mesma pasta:

            ```bash
            docker build -t techchallenge-b .
            docker run --rm -v "%cd%/outputs:/app/outputs" techchallenge-b
            ```
            """
        ).strip()
        + "\n",
        encoding="utf-8",
    )

    dockerfile.write_text(
        textwrap.dedent(
            """
            FROM python:3.11-slim

            WORKDIR /app
            COPY techchallengeB.py /app/techchallengeB.py
            COPY data.csv /app/data.csv

            RUN pip install --no-cache-dir pandas numpy matplotlib scikit-learn shap

            CMD ["python", "techchallengeB.py", "--data", "/app/data.csv", "--output", "/app/outputs"]
            """
        ).strip()
        + "\n",
        encoding="utf-8",
    )

    video_script.write_text(
        textwrap.dedent(
            """
            # Roteiro sugerido para video de demonstracao

            1. Mostrar a estrutura do projeto e o arquivo unico `techchallengeB.py`.
            2. Executar o script com `python techchallengeB.py --data data.csv`.
            3. Abrir a pasta `outputs_techchallenge_b`.
            4. Apresentar graficos de EDA, correlacao e matriz de confusao.
            5. Mostrar as metricas de validacao/teste e explicar a escolha do melhor modelo.
            6. Mostrar feature importance/SHAP e explicar limitacoes.
            7. Reforcar que o sistema e apoio a triagem e que o profissional tem a decisao final.
            """
        ).strip()
        + "\n",
        encoding="utf-8",
    )
    return readme, dockerfile, video_script


def dataframe_to_markdown(df: pd.DataFrame) -> str:
    # Converte DataFrames em Markdown sem depender do pacote opcional tabulate.
    text_df = df.copy()
    for col in text_df.columns:
        if pd.api.types.is_float_dtype(text_df[col]):
            text_df[col] = text_df[col].map(lambda value: "" if pd.isna(value) else f"{value:.4f}")
        else:
            text_df[col] = text_df[col].map(lambda value: "" if pd.isna(value) else str(value))

    headers = [str(col) for col in text_df.columns]
    rows = text_df.values.tolist()
    widths = [
        max(len(headers[index]), *(len(str(row[index])) for row in rows)) if rows else len(headers[index])
        for index in range(len(headers))
    ]

    def render_row(values: Iterable[object]) -> str:
        return "| " + " | ".join(str(value).ljust(widths[index]) for index, value in enumerate(values)) + " |"

    separator = "| " + " | ".join("-" * width for width in widths) + " |"
    lines = [render_row(headers), separator]
    lines.extend(render_row(row) for row in rows)
    return "\n".join(lines)


def format_metrics_table(metrics: pd.DataFrame) -> str:
    # Seleciona e formata as metricas mais relevantes para o relatorio.
    cols = ["dataset", "accuracy", "precision_yes", "recall_yes", "f1_yes", "f1_weighted", "roc_auc"]
    available_cols = [col for col in cols if col in metrics.columns]
    return dataframe_to_markdown(metrics[available_cols].round(4))


def write_markdown_report(
    dataset_path: Path,
    summary: dict,
    validation_metrics: pd.DataFrame,
    best_model_name: str,
    best_metrics: pd.DataFrame,
    feature_importance: pd.DataFrame,
    shap_message: str,
    figures: Iterable[Path],
    docs: Tuple[Path, Path, Path],
    dirs: Dict[str, Path],
) -> Path:
    # Escreve o relatorio tecnico em Markdown com metodologia, metricas e limitacoes.
    top_features = feature_importance.head(15)[["feature", "importance"]].copy()
    top_features["importance"] = top_features["importance"].round(5)
    validation_text = dataframe_to_markdown(validation_metrics.round(4))
    best_text = format_metrics_table(best_metrics)
    features_text = dataframe_to_markdown(top_features)
    figure_lines = "\n".join(f"- `{path}`" for path in figures)
    readme, dockerfile, video_script = docs

    report = f"""
# Relatorio tecnico - Tech Challenge B

## Problema escolhido

O dataset analisado e o arquivo anexado `data.csv`, localizado em `{dataset_path}`.
A tarefa foi formulada como um problema de classificacao binaria: prever se um
caso deve ser marcado como `Dangerous = Yes` ou `Dangerous = No` a partir do
animal e de cinco sintomas observados.

Embora o enunciado use exemplos de diagnostico humano, este dataset representa
um cenario clinico/veterinario de triagem. A solucao deve ser interpretada como
apoio inicial a decisao, nunca como diagnostico final automatico.

## Exploracao dos dados

- Linhas originais: {summary["raw_rows"]}
- Linhas apos limpeza: {summary["clean_rows"]}
- Duplicatas removidas: {summary["duplicates_removed"]}
- Colunas: {", ".join(summary["raw_columns"])}
- Distribuicao do alvo: {summary["target_distribution"]}

Foram gerados graficos de distribuicao do alvo, animais mais frequentes,
sintomas mais frequentes, features numericas derivadas e valores ausentes.

## Pre-processamento

As estrategias utilizadas foram:

- padronizacao de texto com `strip`, caixa baixa e normalizacao de espacos;
- correcao de algumas inconsistencias textuais evidentes, como `seizuers` para `seizures`;
- remocao de duplicatas;
- mapeamento de `Dangerous` para alvo binario;
- preenchimento de sintomas ausentes com `unknown`;
- criacao de features numericas derivadas: quantidade de sintomas unicos,
  quantidade de sintomas desconhecidos e tamanho textual dos sintomas;
- `OneHotEncoder` para variaveis categoricas;
- `StandardScaler` para features numericas;
- pipeline do scikit-learn para evitar vazamento de dados entre treino, validacao e teste.

## Correlacao

A correlacao foi calculada de duas formas:

- correlacao de Pearson entre dummies one-hot e o alvo binario;
- Cramer's V entre cada variavel categorica original e o alvo.

Essas tabelas foram salvas em `tables/encoded_feature_correlations.csv` e
`tables/categorical_association_cramers_v.csv`.

## Modelagem

Foram treinados quatro modelos, cobrindo tecnicas lineares, arvores e metodos
baseados em vizinhanca:

- Regressao Logistica;
- Arvore de Decisao;
- Random Forest;
- KNN.

A separacao foi feita em treino, validacao e teste. O conjunto de validacao foi
usado para escolher o melhor modelo, priorizando `recall` da classe `Yes`, pois
em triagem clinica e mais grave deixar de sinalizar um caso perigoso do que
gerar um alerta falso. O F1-score foi usado como criterio de equilibrio.

### Metricas de validacao

{validation_text}

Modelo selecionado: `{best_model_name}`.

### Metricas do modelo selecionado

{best_text}

## Interpretacao

O script gera importancia de variaveis nativa quando o modelo permite
(`feature_importances_` ou coeficientes). Quando isso nao esta disponivel,
usa permutation importance. SHAP e executado automaticamente quando a biblioteca
esta instalada e compativel com o modelo selecionado.

Status SHAP: {shap_message}

Top features:

{features_text}

## Analise critica

O modelo pode ser util como ferramenta de apoio a triagem, destacando casos que
merecem revisao prioritaria. Entretanto, seu uso pratico exige cuidado:

- o dataset e pequeno e limitado ao universo de animais/sintomas observados;
- categorias novas podem aparecer em producao;
- correlacao nao implica causalidade;
- sintomas textuais foram tratados como categorias, sem contexto clinico amplo;
- e necessario validar o modelo com dados externos e acompanhamento de
  profissionais antes de qualquer uso real.

Assim, a recomendacao e usar a solucao como alerta inicial e ferramenta de
organizacao do fluxo de atendimento. A decisao final deve permanecer com o(a)
medico(a) ou profissional responsavel.

## Artefatos gerados

- Modelo: `models/best_model.pkl`
- Metricas: `tables/validation_metrics.csv` e `tables/best_model_metrics.csv`
- Relatorios de classificacao: `tables/classification_reports.json`
- Exemplo de predicao: `tables/prediction_example.json`
- README: `{readme}`
- Dockerfile: `{dockerfile}`
- Roteiro do video: `{video_script}`

## Figuras

{figure_lines}
"""
    path = dirs["reports"] / "relatorio_tecnico.md"
    path.write_text(textwrap.dedent(report).strip() + "\n", encoding="utf-8")
    return path


def add_text_page(pdf: PdfPages, title: str, body: str) -> None:
    # Adiciona uma pagina textual ao PDF gerado pelo script.
    fig = plt.figure(figsize=(8.27, 11.69))
    fig.patch.set_facecolor("white")
    plt.axis("off")
    wrapped = "\n".join(textwrap.wrap(body, width=92))
    fig.text(0.08, 0.94, title, fontsize=16, weight="bold", va="top")
    fig.text(0.08, 0.88, wrapped, fontsize=10, va="top", linespacing=1.35)
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)


def add_image_page(pdf: PdfPages, image_path: Path, title: str) -> None:
    # Adiciona uma imagem/grafico como pagina no PDF do relatorio.
    if not image_path.exists():
        return
    image = plt.imread(image_path)
    fig = plt.figure(figsize=(8.27, 11.69))
    fig.patch.set_facecolor("white")
    plt.axis("off")
    fig.text(0.08, 0.96, title, fontsize=14, weight="bold", va="top")
    ax = fig.add_axes([0.08, 0.08, 0.84, 0.82])
    ax.imshow(image)
    ax.axis("off")
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)


def write_pdf_report(
    markdown_report: Path,
    figures: Iterable[Path],
    best_metrics: pd.DataFrame,
    dirs: Dict[str, Path],
) -> Path:
    # Converte o relatorio e os principais graficos em um PDF simples.
    path = dirs["reports"] / "relatorio_tecnico.pdf"
    text = markdown_report.read_text(encoding="utf-8")
    plain = text.replace("#", "").replace("`", "")
    chunks = textwrap.wrap(plain, width=4800, replace_whitespace=False)

    with PdfPages(path) as pdf:
        for index, chunk in enumerate(chunks[:4], start=1):
            add_text_page(pdf, f"Relatorio tecnico - parte {index}", chunk)

        metrics_body = best_metrics.round(4).to_string(index=False)
        add_text_page(pdf, "Metricas do modelo selecionado", metrics_body)

        for fig_path in figures:
            add_image_page(pdf, fig_path, fig_path.stem.replace("_", " ").title())

    return path


def save_model(model: Pipeline, dirs: Dict[str, Path], metadata: dict) -> Tuple[Path, Path]:
    # Salva o modelo treinado com pickle e grava metadados sobre a selecao.
    model_path = dirs["models"] / "best_model.pkl"
    metadata_path = dirs["models"] / "model_metadata.json"
    save_json(metadata, metadata_path)

    with model_path.open("wb") as file:
        pickle.dump(model, file)
    return model_path, metadata_path


def main() -> None:
    # Orquestra o pipeline completo: entrada, limpeza, EDA, treino, avaliacao e exportacao.
    args = parse_args()
    dataset_path = resolve_dataset_path(args.data)
    dirs = create_output_dirs(Path(args.output))

    print("Carregando dataset:", dataset_path)
    df_raw = load_dataset(dataset_path)
    df = clean_dataset(df_raw)

    summary = summarize_dataset(df_raw, df, dirs)
    figures = create_eda_plots(df_raw, df, dirs)
    _, corr_figures = correlation_analysis(df, dirs, args.top_n_features)
    figures.extend(corr_figures)

    categorical_cols = ["AnimalName", "symptoms1", "symptoms2", "symptoms3", "symptoms4", "symptoms5"]
    numeric_cols = ["unique_symptom_count", "unknown_symptom_count", "symptom_text_length"]
    feature_cols = categorical_cols + numeric_cols

    split = split_data(
        df,
        feature_cols,
        TARGET_NAME,
        validation_size=args.validation_size,
        test_size=args.test_size,
        random_state=args.random_state,
    )

    split_summary = {
        "train_rows": int(len(split.x_train)),
        "validation_rows": int(len(split.x_val)),
        "test_rows": int(len(split.x_test)),
        "feature_columns": feature_cols,
        "categorical_columns": categorical_cols,
        "numeric_columns": numeric_cols,
    }
    save_json(split_summary, dirs["tables"] / "split_summary.json")

    best_name, best_model, validation_metrics = train_and_select_models(
        split, categorical_cols, numeric_cols, args.random_state, dirs
    )

    best_metrics, reports, confusion_figures = evaluate_best_model(best_name, best_model, split, dirs)
    figures.extend(confusion_figures)

    importance, importance_figures = model_feature_importance(
        best_model, feature_cols, split, dirs, args.top_n_features
    )
    figures.extend(importance_figures)

    shap_path, shap_message = try_generate_shap(
        best_model, split, feature_cols, dirs, args.top_n_features
    )
    if shap_path is not None:
        figures.append(shap_path)

    prediction_example = write_prediction_example(best_model, split, dirs)
    docs = write_readme_and_dockerfile(dirs)

    metadata = {
        "best_model": best_name,
        "dataset_path": str(dataset_path),
        "target": TARGET_NAME,
        "positive_label": "Dangerous=Yes",
        "selection_rule": "Maior recall_yes na validacao; desempate por f1_yes e accuracy.",
        "test_metrics": best_metrics[best_metrics["dataset"] == "test"].to_dict(orient="records"),
        "shap_status": shap_message,
    }
    model_path, metadata_path = save_model(best_model, dirs, metadata)

    markdown_report = write_markdown_report(
        dataset_path=dataset_path,
        summary=summary,
        validation_metrics=validation_metrics,
        best_model_name=best_name,
        best_metrics=best_metrics,
        feature_importance=importance,
        shap_message=shap_message,
        figures=figures,
        docs=docs,
        dirs=dirs,
    )
    pdf_report = write_pdf_report(markdown_report, figures, best_metrics, dirs)

    print("\nPipeline concluido.")
    print(f"Modelo selecionado: {best_name}")
    print("\nMetricas do modelo selecionado:")
    print(best_metrics.round(4).to_string(index=False))
    print("\nExemplo de predicao:")
    print(json.dumps(prediction_example, indent=2, ensure_ascii=False))
    print("\nArtefatos principais:")
    print(f"- Modelo: {model_path}")
    print(f"- Metadados: {metadata_path}")
    print(f"- Relatorio Markdown: {markdown_report}")
    print(f"- Relatorio PDF: {pdf_report}")
    print(f"- README/Dockerfile/Roteiro: {docs[0].parent}")


if __name__ == "__main__":
    main()


Carregando dataset: data.csv
Treinando modelo: logistic_regression
Treinando modelo: decision_tree
Treinando modelo: random_forest
Treinando modelo: knn
Modelo sem importance nativa. Calculando permutation importance no teste.



Pipeline concluido.
Modelo selecionado: knn

Metricas do modelo selecionado:
   dataset model  accuracy  precision_yes  recall_yes  f1_yes  f1_weighted  roc_auc
     train   knn    1.0000         1.0000         1.0  1.0000       1.0000   1.0000
validation   knn    0.9762         0.9762         1.0  0.9880       0.9644   0.7043
      test   knn    0.9821         0.9820         1.0  0.9909       0.9769   0.8552

Exemplo de predicao:
{
  "input": {
    "AnimalName": "bird",
    "symptoms1": "heavy breathing",
    "symptoms2": "yellow in beak",
    "symptoms3": "tail sobbing",
    "symptoms4": "weakness",
    "symptoms5": "sleeping excessively",
    "unique_symptom_count": 5,
    "unknown_symptom_count": 0,
    "symptom_text_length": 73
  },
  "prediction_binary": 1,
  "prediction_label": "Yes - perigoso",
  "probability_yes": 1.0,
  "warning": "A previsao e apenas apoio a triagem. A decisao final deve ser do profissional de saude."
}

Artefatos principais:
- Modelo: outputs_techchallenge

Ver documentação detalhada gerada pelo CODEX AI da OpenAI

In [ ]:
import os

docs_path = "outputs_techchallenge_b"

files_to_show = [
    "reports/relatorio_tecnico.md",
    "docs/README.md",
    "docs/Dockerfile",
    "docs/roteiro_video_demo.md",
    "models/model_metadata.json",
    "tables/dataset_summary.json",
    "tables/classification_reports.json"
]

for file in files_to_show:
    full_path = os.path.join(docs_path, file)
    print("\n" + "="*100)
    print(f"📄 {file}")
    print("="*100)

    if os.path.exists(full_path):
        with open(full_path, "r", encoding="utf-8") as f:
            print(f.read())
    else:
        print("Arquivo não encontrado.")


📄 reports/relatorio_tecnico.md
# Relatorio tecnico - Tech Challenge B

## Problema escolhido

O dataset analisado e o arquivo anexado `data.csv`, localizado em `data.csv`.
A tarefa foi formulada como um problema de classificacao binaria: prever se um
caso deve ser marcado como `Dangerous = Yes` ou `Dangerous = No` a partir do
animal e de cinco sintomas observados.

Embora o enunciado use exemplos de diagnostico humano, este dataset representa
um cenario clinico/veterinario de triagem. A solucao deve ser interpretada como
apoio inicial a decisao, nunca como diagnostico final automatico.

## Exploracao dos dados

- Linhas originais: 869
- Linhas apos limpeza: 839
- Duplicatas removidas: 30
- Colunas: AnimalName, symptoms1, symptoms2, symptoms3, symptoms4, symptoms5, Dangerous
- Distribuicao do alvo: {'yes': 819, 'no': 20}

Foram gerados graficos de distribuicao do alvo, animais mais frequentes,
sintomas mais frequentes, features numericas derivadas e valores ausentes.

## Pre-processame

visualizador dinâmico do RELATORIO EM PDF

In [ ]:
!apt-get update -qq
!apt-get install -y poppler-utils
!pip install -q pdf2image ipywidgets

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpoppler-dev libpoppler-private-dev libpoppler118
Recommended packages:
  poppler-data
The following NEW packages will be installed:
  poppler-utils
The following packages will be upgraded:
  libpoppler-dev libpoppler-private-dev libpoppler118
3 upgraded, 1 newly installed, 0 to remove and 83 not upgraded.
Need to get 1,471 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libpoppler-private-dev amd64 22.02.0-2ubuntu0.13 [198 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libpoppler-dev amd64 22.02.0-2ubuntu0.13 [5,186 B]
Get:3 http

In [ ]:
from pdf2image import convert_from_path
from IPython.display import display
import ipywidgets as widgets

pdf_path = "/content/outputs_techchallenge_b/reports/relatorio_tecnico.pdf"

# qualidade ajustável
zoom = 220
pages = convert_from_path(pdf_path, dpi=zoom)

accordion = widgets.Accordion()

children = []
for i, page in enumerate(pages):
    out = widgets.Output()
    with out:
        display(page)
    children.append(out)

accordion.children = children

for i in range(len(children)):
    accordion.set_title(i, f"Página {i+1}")

display(accordion)

Accordion(children=(Output(), Output(), Output(), Output(), Output(), Output(), Output(), Output(), Output(), …

TESTAR O MODELO

In [ ]:
import pickle
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# carregar modelo
with open("/content/outputs_techchallenge_b/models/best_model.pkl", "rb") as f:
    model = pickle.load(f)

# carregar dataset
df = pd.read_csv("/content/data.csv")

# normalizar colunas (aceita os 2 formatos)
df.columns = [str(c).strip() for c in df.columns]

rename_map = {
    "Animal": "AnimalName",
    "Symptom 1": "symptoms1",
    "Symptom 2": "symptoms2",
    "Symptom 3": "symptoms3",
    "Symptom 4": "symptoms4",
    "Symptom 5": "symptoms5",
}

df = df.rename(columns=rename_map)

symptom_cols = ["symptoms1","symptoms2","symptoms3","symptoms4","symptoms5"]

# listas únicas
animals = sorted(df["AnimalName"].dropna().astype(str).str.lower().unique())

symptoms = sorted(set(
    str(x).strip().lower()
    for col in symptom_cols
    for x in df[col].dropna().unique()
))

if "unknown" not in symptoms:
    symptoms.append("unknown")

# widgets
animal = widgets.Dropdown(
    options=animals,
    description="Animal:"
)

sym_widgets = []
for i in range(5):
    sym_widgets.append(
        widgets.Dropdown(
            options=symptoms,
            description=f"Sintoma {i+1}:"
        )
    )

button = widgets.Button(description="Analisar")
result = widgets.Output()

def on_click(b):
    with result:
        clear_output()

        selected = [w.value for w in sym_widgets]

        # regra para evitar falso positivo sem sintomas
        if all(s == "unknown" for s in selected):
            print("⚠ Nenhum sintoma informado.")
            print("Sem dados suficientes para análise.")
            return

        sample = pd.DataFrame([{
            "AnimalName": animal.value,
            "symptoms1": selected[0],
            "symptoms2": selected[1],
            "symptoms3": selected[2],
            "symptoms4": selected[3],
            "symptoms5": selected[4],
            "unique_symptom_count": len(set(selected)),
            "unknown_symptom_count": selected.count("unknown"),
            "symptom_text_length": len(" ".join(selected))
        }])

        pred = model.predict(sample)[0]
        prob = model.predict_proba(sample)[0][1]

        print("===== RESULTADO =====")
        print("Animal:", animal.value)
        print("Sintomas:", ", ".join(selected))
        print()
        print("Perigoso:", "SIM" if pred == 1 else "NÃO")
        print(f"Probabilidade de risco: {prob:.2%}")

button.on_click(on_click)

display(animal)

for w in sym_widgets:
    display(w)

display(button)
display(result)

Dropdown(description='Animal:', options=('bird', 'black tailed deer', 'buffalo', 'cat', 'cattle', 'cow', 'deer…

Dropdown(description='Sintoma 1:', options=('abdominal detention', 'abdominal pain', 'abnormal behaviour', 'ab…

Dropdown(description='Sintoma 2:', options=('abdominal detention', 'abdominal pain', 'abnormal behaviour', 'ab…

Dropdown(description='Sintoma 3:', options=('abdominal detention', 'abdominal pain', 'abnormal behaviour', 'ab…

Dropdown(description='Sintoma 4:', options=('abdominal detention', 'abdominal pain', 'abnormal behaviour', 'ab…

Dropdown(description='Sintoma 5:', options=('abdominal detention', 'abdominal pain', 'abnormal behaviour', 'ab…

Button(description='Analisar', style=ButtonStyle())

Output()